# 05 · Preprocessing pipeline (incl. PCA dimensionality & HVG sweep)

Runnable, documented front-end to `scripts/preprocessing/run_preprocessing.py`.

Five ordered steps (`STEP_ORDER`): `convert` (HVG filter, `.X`=CPM) → `scgpt` (512-d embedding,
drops OOV genes from `.X`) → `targets` (CTRPv2 `Y_ctrp`/`M_ctrp`) → `splits` (**cell-line-grouped**
`split_ctrp`) → `pca` (**512-d `X_pca`** on the convert counts).

**Decisions baked in:** PCA width **512** to match scGPT (`add_pca.DEFAULT_N_COMPS`); PCA on the
convert counts (full HVG set), not the OOV-dropped `.X`; splits grouped by cell line (viability is
per cell line × drug, so a per-cell split would leak).

In [ ]:
import subprocess, sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.preprocessing.layout import PipelinePaths
from scripts.preprocessing import add_pca

PCA_N_COMPS = 512
print('add_pca default n_comps:', add_pca.DEFAULT_N_COMPS)

## A · Recompute the 512-d PCA baseline (both built variants)

Idempotent re-run of only the PCA step for the already-built variants (the part the 512-d switch
changed). The full pipeline from scratch needs the scGPT venv:
```bash
uv run scripts/preprocessing/run_preprocessing.py --variant <v> --all-drugs \
    --scgpt-python /path/to/scgpt/.venv/bin/python --pca-n-comps 512
```

In [ ]:
BUILT_VARIANTS = ['hvg5000', 'all_genes']
for variant in BUILT_VARIANTS:
    cmd = ['uv', 'run', 'scripts/preprocessing/run_preprocessing.py',
           '--variant', variant, '--start-at', 'pca', '--skip-scgpt', '--force-pca',
           '--pca-n-comps', str(PCA_N_COMPS)]
    print('\n' + '=' * 70 + f'\n{variant}\n' + '=' * 70)
    proc = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    print(proc.stdout[-1500:])
    if proc.returncode != 0:
        print('STDERR:\n', proc.stderr[-1500:]); raise RuntimeError(variant)

In [ ]:
import scanpy as sc
for variant in BUILT_VARIANTS:
    p = PipelinePaths.build(None, variant)
    a = sc.read_h5ad(p.targets_h5ad)
    print(f'{variant:9s} | X_pca {a.obsm["X_pca"].shape} | X_scGPT {a.obsm["X_scGPT"].shape} '
          f'| K={len(a.uns.get("ctrp_drugs", []))}')
    assert a.obsm['X_pca'].shape[1] == PCA_N_COMPS

## B · HVG-count sweep — generate the 1k / 2k / 3k / 5k gene-set variants

Builds an independent variant per HVG count so `07_training.ipynb` can find scGPT's filtering
**sweet spot** under cross-validation. Each variant runs the **full** pipeline incl. **scGPT
re-embedding** — this is **expensive (hours + GBs)**, so it is gated behind `RUN_HVG_SWEEP`.
`hvg5000` already exists and is reused. Idempotent: variants whose targets file exists are skipped
(pass `--overwrite` to force).

Set `RUN_HVG_SWEEP = True` and a valid `SCGPT_PYTHON` to run. Equivalent CLI per variant:
```bash
uv run scripts/preprocessing/run_preprocessing.py --variant hvg2000 --all-drugs \
    --scgpt-python <SCGPT_PYTHON> --pca-n-comps 512
```

In [ ]:
RUN_HVG_SWEEP = False                                  # flip to True to (re)build the sweep variants
SCGPT_PYTHON  = '/Users/selin/PycharmProjects/scGPT/.venv/bin/python'
SWEEP_VARIANTS = ['hvg1000', 'hvg2000', 'hvg3000']     # hvg5000 already built

if not RUN_HVG_SWEEP:
    print('RUN_HVG_SWEEP is False — skipping the heavy scGPT sweep build.')
    for v in SWEEP_VARIANTS:
        p = PipelinePaths.build(None, v)
        print(f'  {v}: {"present" if Path(p.targets_h5ad).exists() else "MISSING"}')
else:
    for variant in SWEEP_VARIANTS:
        p = PipelinePaths.build(None, variant)
        if Path(p.targets_h5ad).exists():
            print(f'{variant}: targets present, skipping.'); continue
        cmd = ['uv', 'run', 'scripts/preprocessing/run_preprocessing.py',
               '--variant', variant, '--all-drugs',
               '--scgpt-python', SCGPT_PYTHON, '--pca-n-comps', str(PCA_N_COMPS)]
        print('\n' + '=' * 70 + f'\n{variant}: {" ".join(cmd)}\n' + '=' * 70)
        proc = subprocess.run(cmd, cwd=ROOT, text=True)
        if proc.returncode != 0:
            raise RuntimeError(f'{variant} failed (exit {proc.returncode})')